# Nested Cross Validation - Starter Notebook
This notebook demonstrates nested cross validation where hyperparameter tuning is done inside each outer fold.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [ ]:
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="species")

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC())
])

param_grid = {
    "svc__C": [0.1, 1, 10],
    "svc__kernel": ["linear", "rbf"],
    "svc__gamma": ["scale", "auto"]
}

X.head()

In [ ]:
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=inner_cv,
    scoring="accuracy",
    n_jobs=-1
)

nested_scores = cross_val_score(grid, X, y, cv=outer_cv, scoring="accuracy", n_jobs=-1)

print("Nested CV scores:", nested_scores)
print(f"Mean nested accuracy: {nested_scores.mean():.3f}")

In [ ]:
grid.fit(X, y)

print("Best params from full-data tuning:")
print(grid.best_params_)

summary = pd.DataFrame({
    "nested_mean_accuracy": [nested_scores.mean()],
    "nested_std_accuracy": [nested_scores.std()]
})
summary